# 01. Data Exploration - Fashion Product Images

Kaggle Fashion Product Images (Small) 데이터셋 탐색
- 전체 이미지 수 및 카테고리별 분포
- 클래스 불균형 확인
- 샘플 이미지 확인
- 우리 프로젝트에 사용할 라벨 매핑 설계

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
from collections import Counter

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_ROOT = os.path.expanduser('~/.cache/kagglehub/datasets/paramaggarwal/fashion-product-images-small/versions/1')
IMG_DIR = os.path.join(DATA_ROOT, 'images')
CSV_PATH = os.path.join(DATA_ROOT, 'styles.csv')

## 1. 메타데이터 로드 및 기본 정보

In [ ]:
df = pd.read_csv(CSV_PATH, on_bad_lines='skip')
print(f'전체 레코드 수: {len(df)}')
print(f'컬럼: {df.columns.tolist()}')
print(f'\n결측치:')
print(df.isnull().sum())
df.head()

## 2. 실제 이미지 존재 여부 확인
메타데이터에는 있지만 이미지 파일이 없는 경우 제거

In [ ]:
df['image_path'] = df['id'].apply(lambda x: os.path.join(IMG_DIR, f'{x}.jpg'))
df['image_exists'] = df['image_path'].apply(os.path.exists)
print(f'이미지 존재: {df["image_exists"].sum()} / {len(df)}')
print(f'이미지 없음: {(~df["image_exists"]).sum()}')
df = df[df['image_exists']].copy()
print(f'\n정제 후 레코드 수: {len(df)}')

## 3. articleType 분포 (상위 25개)

In [ ]:
top25 = df['articleType'].value_counts().head(25)

fig, ax = plt.subplots(figsize=(14, 7))
top25.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Count')
ax.set_title('Top 25 Article Types')
ax.invert_yaxis()
for i, v in enumerate(top25.values):
    ax.text(v + 30, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 4. 우리 프로젝트용 라벨 매핑

44K+ 이미지 중 의류/신발 카테고리만 선별하여 10개 클래스로 매핑

In [ ]:
# articleType → 우리 라벨 매핑
LABEL_MAP = {
    # 상의 (Tops)
    'Tshirts': 'T-shirt',
    'Shirts': 'Shirt',
    'Tops': 'T-shirt',           # Tops → T-shirt로 병합
    'Sweatshirts': 'Hoodie',
    
    # 아우터 (Outerwear)
    'Jackets': 'Jacket',
    'Blazers': 'Jacket',         # Blazers → Jacket 병합
    'Coats': 'Jacket',           # Coats → Jacket 병합
    'Sweaters': 'Hoodie',        # Sweaters → Hoodie 병합
    
    # 하의 (Bottoms)
    'Jeans': 'Jeans',
    'Trousers': 'Pants',
    'Shorts': 'Pants',           # Shorts → Pants 병합
    'Track Pants': 'Pants',
    'Skirts': 'Skirt',
    
    # 원피스
    'Dresses': 'Dress',
    'Jumpsuit': 'Dress',         # Jumpsuit → Dress 병합
    
    # 신발 (Shoes)
    'Casual Shoes': 'Sneakers',
    'Sports Shoes': 'Sneakers',
    'Formal Shoes': 'Boots',     # Formal → Boots 계열
    'Heels': 'Boots',
    'Flats': 'Sneakers',
}

df['label'] = df['articleType'].map(LABEL_MAP)
df_filtered = df[df['label'].notna()].copy()

print(f'매핑된 레코드 수: {len(df_filtered)} / {len(df)}')
print(f'\n라벨별 분포:')
print(df_filtered['label'].value_counts().to_string())

In [ ]:
# 매핑 후 분포 시각화
label_counts = df_filtered['label'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7',
          '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9']
label_counts.plot(kind='bar', ax=ax, color=colors[:len(label_counts)])
ax.set_ylabel('Count')
ax.set_title('Our Label Distribution (10 Classes)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
for i, v in enumerate(label_counts.values):
    ax.text(i, v + 50, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 5. 색상 분포 및 매핑

In [ ]:
COLOR_MAP = {
    'Black': 'black',
    'White': 'white',
    'Grey': 'gray', 'Grey Melange': 'gray', 'Charcoal': 'gray', 'Silver': 'gray',
    'Blue': 'blue', 'Navy Blue': 'blue', 'Teal': 'blue', 'Turquoise Blue': 'blue',
    'Red': 'red', 'Maroon': 'red', 'Burgundy': 'red', 'Rust': 'red', 'Magenta': 'red',
    'Beige': 'beige', 'Cream': 'beige', 'Skin': 'beige', 'Nude': 'beige',
    'Khaki': 'beige', 'Tan': 'beige', 'Gold': 'beige', 'Mushroom Brown': 'beige',
    'Brown': 'beige', 'Coffee Brown': 'beige', 'Taupe': 'beige',
    'Green': 'green', 'Olive': 'green', 'Lime Green': 'green', 'Sea Green': 'green',
    'Fluorescent Green': 'green',
}

df_filtered['color_label'] = df_filtered['baseColour'].map(COLOR_MAP)
print(f'색상 매핑 성공: {df_filtered["color_label"].notna().sum()} / {len(df_filtered)}')
print(f'\n매핑 안 된 색상:')
unmapped = df_filtered[df_filtered['color_label'].isna()]['baseColour'].value_counts()
print(unmapped.to_string())
print(f'\n색상별 분포:')
print(df_filtered['color_label'].value_counts().to_string())

In [ ]:
# 색상 분포 시각화
color_counts = df_filtered['color_label'].value_counts()
color_hex = {'black': '#2C3E50', 'white': '#ECF0F1', 'gray': '#95A5A6',
             'blue': '#3498DB', 'red': '#E74C3C', 'beige': '#D4A574', 'green': '#27AE60'}

fig, ax = plt.subplots(figsize=(8, 5))
bars = color_counts.plot(kind='bar', ax=ax, 
    color=[color_hex.get(c, '#999') for c in color_counts.index],
    edgecolor='black', linewidth=0.5)
ax.set_ylabel('Count')
ax.set_title('Color Distribution (7 Groups)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
for i, v in enumerate(color_counts.values):
    ax.text(i, v + 30, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 6. 스타일(usage) 분포

In [ ]:
STYLE_MAP = {
    'Casual': 'casual',
    'Smart Casual': 'casual',
    'Formal': 'formal',
    'Party': 'formal',
    'Sports': 'sporty',
    'Ethnic': 'casual',
    'Travel': 'casual',
    'Home': 'casual',
}

df_filtered['style_label'] = df_filtered['usage'].map(STYLE_MAP)
print(f'스타일 매핑 성공: {df_filtered["style_label"].notna().sum()} / {len(df_filtered)}')
print(f'\n스타일별 분포:')
print(df_filtered['style_label'].value_counts().to_string())

## 7. 샘플 이미지 Grid
각 카테고리별 샘플 3장씩 확인

In [ ]:
labels = sorted(df_filtered['label'].unique())
n_labels = len(labels)
samples_per_label = 3

fig, axes = plt.subplots(n_labels, samples_per_label, figsize=(3*samples_per_label, 3*n_labels))

for i, label in enumerate(labels):
    subset = df_filtered[df_filtered['label'] == label].sample(n=samples_per_label, random_state=42)
    for j, (_, row) in enumerate(subset.iterrows()):
        img = Image.open(row['image_path']).resize((224, 224))
        axes[i][j].imshow(img)
        axes[i][j].axis('off')
        if j == 0:
            axes[i][j].set_title(f'{label}', fontsize=12, fontweight='bold')

plt.suptitle('Sample Images per Category', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. 최종 데이터 요약

In [ ]:
# 색상과 스타일이 모두 매핑된 데이터만 최종 사용
df_final = df_filtered[df_filtered['color_label'].notna() & df_filtered['style_label'].notna()].copy()

print('='*50)
print('최종 학습 데이터 요약')
print('='*50)
print(f'전체 이미지 수: {len(df_final)}')
print(f'카테고리 수: {df_final["label"].nunique()}')
print(f'색상 그룹 수: {df_final["color_label"].nunique()}')
print(f'스타일 수: {df_final["style_label"].nunique()}')
print(f'\n카테고리별 분포:')
print(df_final['label'].value_counts().to_string())
print(f'\n색상별 분포:')
print(df_final['color_label'].value_counts().to_string())
print(f'\n스타일별 분포:')
print(df_final['style_label'].value_counts().to_string())

In [ ]:
# 최종 데이터프레임 저장
save_cols = ['id', 'image_path', 'label', 'color_label', 'style_label', 'articleType', 'baseColour', 'usage', 'gender', 'season']
df_final[save_cols].to_csv('/Users/parkyoungbin/Desktop/ml2/model/data/dataset_final.csv', index=False)
print('dataset_final.csv 저장 완료')